# Task 2: Retrieval-Augmented Generation (RAG) From Scratch
**Applied Generative AI Engineering & System Optimization**
*Corpus: 335 Pages across 10 Landmark arXiv Papers | Zero LangChain / LlamaIndex*

This notebook executes:
1. Automated download & text extraction of 10 landmark arXiv PDFs (335 pages, 169,731 words).
2. Implementation of Pure Python Okapi BM25 and FAISS IndexFlatIP Dense Retriever.
3. Hybrid Score Fusion with convex normalization.
4. Evaluation of 15 hand-authored test questions against ground-truth page excerpts.
5. Chunk size benchmark: Setting A (256 tokens) vs Setting B (512 tokens) and Vector Dilution analysis.
6. Grounded answer generation with citations using Qwen2.5-0.5B-Instruct.
7. Theoretical explanations of BM25 vs Dense and Cross-Encoder Re-rankers in own words.


In [ ]:
# 1. Environment Setup & Dependencies
!pip install -q pypdf pymupdf sentence-transformers faiss-cpu transformers torch

import os
import json
import math
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


## 2. Build Real 10-Paper Academic Corpus (335 Pages)
Downloads 10 seminal papers directly from arXiv:
- Attention Is All You Need (1706.03762)
- LoRA (2106.09685)
- QLoRA (2305.14314)
- FlashAttention (2205.14135)
- ResNet (1512.03385)
- BERT (1810.04805)
- Chinchilla (2203.15556)
- GPT-3 (2005.14165)
- Llama 2 (2307.09288)
- DDPM (2006.11239)


In [ ]:
# Download and parse PDFs into structured sections
!python ../task2_rag/build_corpus_real.py


## 3. Benchmark 15 Technical Questions
15 hand-authored technical questions targeting subtle equations, algorithmic choices, footnotes, and theorem proofs with exact ground-truth excerpts.


In [ ]:
# Verify 15 benchmark questions
!python ../task2_rag/build_15_questions.py


## 4. Chunk Size Evaluation: Setting A (256 Tokens) vs Setting B (512 Tokens)
Evaluates Recall@1, Recall@3, Recall@5, and MRR@5 across Sparse BM25, Dense FAISS, and Hybrid modes.


In [ ]:
# Run retrieval benchmark
!python ../task2_rag/evaluate_rag.py


## 5. End-to-End Grounded Question Answering with Citations
Generates factual answers grounded strictly in retrieved passages using Qwen2.5-0.5B-Instruct.


In [ ]:
# Generate grounded answers
!python ../task2_rag/generate_qa_answers.py


## 6. Retrieval Concepts in Own Words
- **What BM25 Adds to Embedding Search**: Dense bi-encoders map text into a continuous semantic latent space. While brilliant at matching concepts with different phrasing (*"speed up training"* $\leftrightarrow$ *"accelerate throughput"*), bi-encoders fail on exact out-of-vocabulary tokens, version strings (`v2.1` vs `v2.2`), mathematical formulas, and acronyms (`RoBERTa` vs `BERT`). BM25 introduces term-frequency and inverse-document-frequency exactness. It acts as an indispensable lexical anchor, ensuring rare identifiers are matched with exact precision rather than smoothed away into generic semantic clusters.
- **What a Re-Ranker Does**: Dense retrieval relies on bi-encoders that encode query and passage independently into single vectors ($sim = \mathbf{q}^T \mathbf{p}$), enabling fast sub-millisecond MIPS searches over millions of chunks at the cost of cross-token interaction. A cross-encoder re-ranker takes the top-20 retrieved candidates, feeds the concatenated string `[CLS] Query [SEP] Passage [SEP]` through full multi-head self-attention, and scores relevance with all-to-all token attention. This captures subtle syntax, token ordering, and logical negations that bi-encoders completely miss.
